# PubMedQA：生成式 LoRA-SFT vs 判别式 PubMedBERT

在**完全相同的测试集**上对比四个模型的 Yes / No / Maybe 三分类表现：

| # | 模型 | 来源 |
|---|---|---|
| 1 | PubMedBERT baseline（无类别权重，按 val acc 选型） | 已有工作 |
| 2 | PubMedBERT improved（类别权重，按 val macro-F1 选型） | 已有工作 |
| 3 | Qwen2.5-Instruct 未微调 | 本 notebook |
| 4 | Qwen2.5-Instruct + LoRA-SFT | 本 notebook |

划分、随机种子、选型协议**全部沿用 `train_pubmedbert.py`**：两阶段分层划分，`seed=42`，
train 700 / val 150 / test 150，验证集选型，测试集只评一次。


## 1. 安装依赖

In [1]:
!pip -q install -U "transformers>=4.44" "peft>=0.11" "datasets>=2.20" "accelerate>=0.33" scikit-learn
print("done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 61.5 MB/s eta 0:00:00
done


## 2. 配置

与 `train_pubmedbert.py` 对齐的超参用注释标出。

In [2]:
import os, json, random, time
import numpy as np, torch
import torch.nn.functional as F

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"   # 显存够可换 Qwen/Qwen2.5-1.5B-Instruct

RUN_MODE      = "improved"    # "baseline" 或 "improved"，命名与 BERT 脚本一致
USE_CLASS_WEIGHT = (RUN_MODE == "improved")
SELECT_BY     = "macro_f1" if RUN_MODE == "improved" else "accuracy"

RANDOM_SEED   = 42            # ← 与 BERT 脚本相同
VAL_SIZE      = 0.15          # ← 相同
TEST_SIZE     = 0.15          # ← 相同
EPOCHS        = 3
BATCH_SIZE    = 2             # 显存不足就保持 2，用梯度累积撑有效 batch
GRAD_ACC      = 8             # 有效 batch = 16
LR            = 2e-4          # LoRA 用的学习率比全量微调大
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1
GRAD_CLIP     = 1.0
MAX_CTX_WORDS = 350           # 上下文预算，与 BERT 的 512 token 大致对齐

LABEL_NAMES = ["yes", "no", "maybe"]
LABEL_MAP   = {"yes": 0, "no": 1, "maybe": 2}

random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = (DEVICE == "cuda")
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if use_amp else "CPU（会很慢，请改运行时为 T4）")
print("模式  :", RUN_MODE, " 类别权重:", USE_CLASS_WEIGHT, " 选型指标:", SELECT_BY)

device: cuda | Tesla T4
模式  : improved  类别权重: True  选型指标: macro_f1


## 3. 数据：复刻 `train_pubmedbert.py` 的划分

下面这段划分代码是从 `train_pubmedbert.py` **原样搬过来的**——同样的两阶段
`train_test_split`、同样的 `random_state=42`、同样的分层依据。因此得到的
train / val / test 索引与 BERT 实验**逐样本一致**，两个模型评的是同一批 150 条测试样本。

跑完检查输出是否为 `700 / 150 / 150`。

In [3]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from collections import Counter

try:
    dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
except Exception:
    dataset = load_dataset("pubmed_qa", "pqa_labeled", split="train")
print("样本数  :", len(dataset))
print("标签分布:", dict(Counter(s["final_decision"] for s in dataset)))

questions = [s["question"].strip() for s in dataset]
passages  = [" ".join(s["context"]["contexts"]) for s in dataset]
labels    = [LABEL_MAP[s["final_decision"].lower()] for s in dataset]
labels_arr = np.array(labels)

# ↓↓↓ 与 train_pubmedbert.py 完全一致 ↓↓↓
trainval_idx, test_idx = train_test_split(
    np.arange(len(questions)), test_size=TEST_SIZE,
    random_state=RANDOM_SEED, stratify=labels_arr,
)
val_ratio = VAL_SIZE / (1.0 - TEST_SIZE)
train_idx, val_idx = train_test_split(
    trainval_idx, test_size=val_ratio,
    random_state=RANDOM_SEED, stratify=labels_arr[trainval_idx],
)
# ↑↑↑ 与 train_pubmedbert.py 完全一致 ↑↑↑

print(f"训练集 : {len(train_idx)}  |  验证集 : {len(val_idx)}  |  测试集 : {len(test_idx)}")
assert (len(train_idx), len(val_idx), len(test_idx)) == (700, 150, 150), "划分对不上，检查数据源"
print("测试集类别分布:", {LABEL_NAMES[k]: v for k, v in sorted(Counter(labels_arr[test_idx]).items())})
print("→ 应为 {'yes': 83, 'no': 51, 'maybe': 16}，与 BERT 结果文件一致")

README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_labeled/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.08MB            

pqa_labeled/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

样本数  : 1000
标签分布: {'yes': 552, 'no': 338, 'maybe': 110}
训练集 : 700  |  验证集 : 150  |  测试集 : 150
测试集类别分布: {'yes': 83, 'no': 51, 'maybe': 16}
→ 应为 {'yes': 83, 'no': 51, 'maybe': 16}，与 BERT 结果文件一致


## 4. 指令格式

BERT 的输入是 `question [SEP] passages`，靠分类头输出三类 logits。
生成式模型没有分类头，所以改成自然语言指令 + chat template，让模型「说出」标签。

信息内容相同（question + contexts），上下文预算也大致对齐 BERT 的 512 token。

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def build_prompt(i):
    ctx = " ".join(passages[i].split()[:MAX_CTX_WORDS])
    user = (f"Context: {ctx}\n\n"
            f"Question: {questions[i]}\n\n"
            "Based only on the context above, answer with exactly one word: yes, no, or maybe.")
    return tok.apply_chat_template([{"role": "user", "content": user}],
                                   tokenize=False, add_generation_prompt=True)

demo = build_prompt(train_idx[0])
print(demo[:500], "\n ...\n")
print("LABEL:", LABEL_NAMES[labels[train_idx[0]]])
print("prompt tokens:", len(tok(demo, add_special_tokens=False)["input_ids"]))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Context: Testosterone measurement by liquid chromatography tandem mass spectrometry (LC-MS/MS) is well accepted as the preferred technique for the analysis of testosterone. Variation is seen between assays and this may be due to differences in calibration as commercial calibrators for this assay are not readily available. We investigated the effects calibration in routine clinical L 
 ...

LABEL: no
prompt tokens: 298


In [5]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16 if use_amp else torch.float32,
).to(DEVICE)
print(model.config.model_type, f"{sum(p.numel() for p in model.parameters())/1e6:.0f}M params")

LABEL_TOK = [tok(l, add_special_tokens=False)["input_ids"] for l in LABEL_NAMES]
print("标签分词:", {l: t for l, t in zip(LABEL_NAMES, LABEL_TOK)})

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

qwen2 494M params
标签分词: {'yes': [9693], 'no': [2152], 'maybe': [36760]}


## 5. 评测：受限打分，而非自由生成

**这是一个刻意的设计决定，也是本项目和直接跑 generate 的关键差别。**

自由生成再解析输出时，小模型经常不遵循「只回答一个词」的指令（输出 `Yes.`、
`The answer is yes`、或一整段解释），解析失败会被计成错误答案——
**评测里就混进了格式不遵循的噪声，测的不再是分类能力**。

改为：对 `yes` / `no` / `maybe` 三个候选分别计算其在当前 prompt 下的序列对数似然，取最大者。
每个样本必然得到合法标签，测量的是模型对三个选项的相对偏好，
与 BERT 的 softmax 三分类在语义上对齐，指标可比。

In [6]:
@torch.no_grad()
def predict_one(m, i):
    p_ids = tok(build_prompt(i), add_special_tokens=False)["input_ids"]
    seqs = [p_ids + a for a in LABEL_TOK]
    alen = [len(a) for a in LABEL_TOK]
    L = max(len(s) for s in seqs)
    ids  = torch.full((3, L), tok.pad_token_id, dtype=torch.long)
    attn = torch.zeros((3, L), dtype=torch.long)
    for k, s in enumerate(seqs):
        ids[k, :len(s)] = torch.tensor(s); attn[k, :len(s)] = 1
    ids, attn = ids.to(DEVICE), attn.to(DEVICE)
    lp = F.log_softmax(m(input_ids=ids, attention_mask=attn).logits.float(), dim=-1)
    scores = []
    for k, s in enumerate(seqs):
        tot, st = 0.0, len(s) - alen[k]
        for pos in range(st, len(s)):
            tot += lp[k, pos - 1, ids[k, pos]].item()
        scores.append(tot)
    return int(np.argmax(scores))


from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def run_eval(m, idxs):
    m.eval()
    preds = [predict_one(m, i) for i in idxs]
    trues = [labels[i] for i in idxs]
    return np.array(trues), np.array(preds)

def metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "per_class_f1": f1_score(y_true, y_pred, average=None, labels=[0,1,2], zero_division=0).tolist(),
    }

def show(tag, y_true, y_pred):
    mt = metrics(y_true, y_pred)
    print(f"\n==== {tag} ====")
    print(f"Accuracy : {mt['accuracy']:.4f}   Macro F1 : {mt['macro_f1']:.4f}   Weighted F1 : {mt['weighted_f1']:.4f}")
    print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4, zero_division=0))
    print("Confusion matrix (rows=true, cols=pred):")
    print("          " + "  ".join(f"{l:>7}" for l in LABEL_NAMES))
    for r, row in enumerate(confusion_matrix(y_true, y_pred, labels=[0,1,2])):
        print(f"  {LABEL_NAMES[r]:>5}  " + "  ".join(f"{v:7d}" for v in row))
    return mt

## 6. Baseline：**微调前**先在测试集上测一次

没有这条线，后面任何「提升」都无从谈起。

In [7]:
t0 = time.time()
yt, yp = run_eval(model, test_idx)
res_qwen_base = show(f"{MODEL_NAME}  未微调  [TEST]", yt, yp)
print(f"\n用时 {time.time()-t0:.0f}s")


==== Qwen/Qwen2.5-0.5B-Instruct  未微调  [TEST] ====
Accuracy : 0.1067   Macro F1 : 0.0643   Weighted F1 : 0.0206
              precision    recall  f1-score   support

         yes     0.0000    0.0000    0.0000        83
          no     0.0000    0.0000    0.0000        51
       maybe     0.1067    1.0000    0.1928        16

    accuracy                         0.1067       150
   macro avg     0.0356    0.3333    0.0643       150
weighted avg     0.0114    0.1067    0.0206       150

Confusion matrix (rows=true, cols=pred):
              yes       no    maybe
    yes        0        0       83
     no        0        0       51
  maybe        0        0       16

用时 24s


## 7. LoRA 配置

In [9]:
import sys
import peft.import_utils as _piu
_piu.is_torchao_available = lambda: False
for _n, _m in list(sys.modules.items()):
    if _n.startswith("peft") and hasattr(_m, "is_torchao_available"):
        _m.is_torchao_available = lambda: False
print("已绕过 torchao 检测")

已绕过 torchao 检测


In [10]:
from peft import LoraConfig, get_peft_model

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
))
model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


## 8. 训练数据：只在答案 token 上算 loss

prompt 部分全部 mask 成 `-100`——要学的是「给定上下文该回答什么」，不是把上下文背下来。

`improved` 模式下按类别频率给每条样本加权，与 BERT 脚本里的
`CrossEntropyLoss(weight=...)` 是同一个思路，只是这里作用在样本级。

In [12]:
from torch.utils.data import Dataset, DataLoader

train_counts = Counter(labels[i] for i in train_idx)
print("训练集类别分布:", {LABEL_NAMES[k]: v for k, v in sorted(train_counts.items())})

if USE_CLASS_WEIGHT:
    cw = [len(train_idx) / (3 * train_counts[c]) for c in range(3)]
    print("类别权重:", {LABEL_NAMES[c]: round(cw[c], 3) for c in range(3)})
else:
    cw = [1.0, 1.0, 1.0]

class SFTData(Dataset):
    def __init__(self, idxs):
        self.items = []
        for i in idxs:
            p = tok(build_prompt(i), add_special_tokens=False)["input_ids"]
            a = LABEL_TOK[labels[i]] + [tok.eos_token_id]
            self.items.append({"input_ids": p + a,
                               "labels": [-100]*len(p) + a,
                               "w": cw[labels[i]]})
    def __len__(self):  return len(self.items)
    def __getitem__(self, k): return self.items[k]

def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    ids  = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
    lab  = torch.full((len(batch), L), -100, dtype=torch.long)
    attn = torch.zeros((len(batch), L), dtype=torch.long)
    for k, b in enumerate(batch):
        n = len(b["input_ids"])
        ids[k, :n]  = torch.tensor(b["input_ids"])
        lab[k, :n]  = torch.tensor(b["labels"])
        attn[k, :n] = 1
    w = torch.tensor([b["w"] for b in batch], dtype=torch.float)
    return {"input_ids": ids, "labels": lab, "attention_mask": attn, "w": w}

train_loader = DataLoader(SFTData(train_idx), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
print("train batches:", len(train_loader))

训练集类别分布: {'yes': 386, 'no': 236, 'maybe': 78}
类别权重: {'yes': 0.604, 'no': 0.989, 'maybe': 2.991}
train batches: 350


## 9. 训练：每个 epoch 用验证集选型

训练循环刻意写成和 `train_pubmedbert.py` 一样的结构：每个 epoch 结束在验证集上评一次，
按 `SELECT_BY` 保留最优权重，**测试集全程不参与**。这样两个实验的选型协议一致，
最终数字才真的可比。

In [13]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = (len(train_loader) // GRAD_ACC) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps*WARMUP_RATIO), max(total_steps,1))
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def batch_loss(out_logits, lab, w):
    lg  = out_logits[:, :-1, :]
    tgt = lab[:, 1:]
    lt  = F.cross_entropy(lg.reshape(-1, lg.size(-1)).float(), tgt.reshape(-1),
                          reduction="none", ignore_index=-100).view(tgt.shape)
    m   = (tgt != -100).float()
    per_ex = (lt * m).sum(1) / m.sum(1).clamp(min=1)
    return (per_ex * w).sum() / w.sum()

history, best_score, best_state, best_epoch = [], -1.0, None, -1

for epoch in range(1, EPOCHS + 1):
    model.train(); ep_loss, nb = 0.0, 0
    optimizer.zero_grad()
    for step, b in enumerate(train_loader, 1):
        ids, lab = b["input_ids"].to(DEVICE), b["labels"].to(DEVICE)
        attn, w  = b["attention_mask"].to(DEVICE), b["w"].to(DEVICE)
        with torch.amp.autocast("cuda", enabled=use_amp):
            logits = model(input_ids=ids, attention_mask=attn).logits
        loss = batch_loss(logits, lab, w) / GRAD_ACC
        scaler.scale(loss).backward()
        if step % GRAD_ACC == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            optimizer.zero_grad()
        ep_loss += loss.item() * GRAD_ACC; nb += 1
        if step % 100 == 0:
            print(f"  epoch {epoch} step {step}/{len(train_loader)} loss={ep_loss/nb:.4f}")

    yt, yp = run_eval(model, val_idx)
    mt = metrics(yt, yp)
    pc = mt["per_class_f1"]
    print(f"Epoch {epoch}/{EPOCHS}  loss={ep_loss/nb:.4f}  val_acc={mt['accuracy']:.4f}  "
          f"val_macro_f1={mt['macro_f1']:.4f}  [yes={pc[0]:.3f} no={pc[1]:.3f} maybe={pc[2]:.3f}]")
    history.append({"epoch": epoch, "val_acc": mt["accuracy"], "val_macro_f1": mt["macro_f1"],
                    "f1_yes": pc[0], "f1_no": pc[1], "f1_maybe": pc[2]})

    score = mt["macro_f1"] if SELECT_BY == "macro_f1" else mt["accuracy"]
    if score > best_score:
        best_score, best_epoch = score, epoch
        best_state = {k: v.detach().cpu().clone()
                      for k, v in model.state_dict().items() if "lora" in k.lower()}
        print(f"  -> 新最优 (val {SELECT_BY}={best_score:.4f})")

print(f"\n按验证集选中 epoch {best_epoch}")

/tmp/ipykernel_2906/569942212.py:34: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(optimizer); scaler.update(); scheduler.step()


  epoch 1 step 100/350 loss=1.0031
  epoch 1 step 200/350 loss=0.7371
  epoch 1 step 300/350 loss=0.6654
Epoch 1/3  loss=0.6362  val_acc=0.6733  val_macro_f1=0.4648  [yes=0.753 no=0.642 maybe=0.000]
  -> 新最优 (val macro_f1=0.4648)
  epoch 2 step 100/350 loss=0.4047
  epoch 2 step 200/350 loss=0.3585
  epoch 2 step 300/350 loss=0.3804
Epoch 2/3  loss=0.3770  val_acc=0.6400  val_macro_f1=0.4910  [yes=0.719 no=0.690 maybe=0.065]
  -> 新最优 (val macro_f1=0.4910)
  epoch 3 step 100/350 loss=0.2284
  epoch 3 step 200/350 loss=0.2058
  epoch 3 step 300/350 loss=0.2128
Epoch 3/3  loss=0.2199  val_acc=0.6533  val_macro_f1=0.4997  [yes=0.731 no=0.702 maybe=0.067]
  -> 新最优 (val macro_f1=0.4997)

按验证集选中 epoch 3


## 10. 用选中的权重，在测试集上评一次

In [14]:
missing = model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()}, strict=False)
yt, yp = run_eval(model, test_idx)
res_qwen_sft = show(f"{MODEL_NAME} + LoRA-SFT ({RUN_MODE})  [TEST]", yt, yp)


==== Qwen/Qwen2.5-0.5B-Instruct + LoRA-SFT (improved)  [TEST] ====
Accuracy : 0.6133   Macro F1 : 0.4513   Weighted F1 : 0.6125
              precision    recall  f1-score   support

         yes     0.8500    0.6145    0.7133        83
          no     0.5325    0.8039    0.6406        51
       maybe     0.0000    0.0000    0.0000        16

    accuracy                         0.6133       150
   macro avg     0.4608    0.4728    0.4513       150
weighted avg     0.6514    0.6133    0.6125       150

Confusion matrix (rows=true, cols=pred):
              yes       no    maybe
    yes       51       25        7
     no        4       41        6
  maybe        5       11        0


## 11. 四方对比

PubMedBERT 两行的数字直接来自 `results_baseline.txt` / `results_improved.txt`，
**同一测试集、同一指标**，可以直接并排。

In [15]:
BERT_BASELINE = {"tag": "PubMedBERT baseline (110M, 判别式)", "params": "110M",
                 "accuracy": 0.6533, "macro_f1": 0.4439, "weighted_f1": 0.6119,
                 "per_class_f1": [0.7459, 0.5859, 0.0000]}
BERT_IMPROVED = {"tag": "PubMedBERT improved (110M, 判别式)", "params": "110M",
                 "accuracy": 0.6533, "macro_f1": 0.4856, "weighted_f1": 0.6252,
                 "per_class_f1": [0.7303, 0.6154, 0.1111]}

short = MODEL_NAME.split("/")[-1]
res_qwen_base["tag"] = f"{short} 未微调 (生成式)"
res_qwen_sft["tag"]  = f"{short} + LoRA-SFT [{RUN_MODE}] (生成式)"
for r in (res_qwen_base, res_qwen_sft):
    r["params"] = "0.5B" if "0.5B" in short else "1.5B"

rows = [BERT_BASELINE, BERT_IMPROVED, res_qwen_base, res_qwen_sft]

print("\n| 模型 | 参数量 | accuracy | macro-F1 | weighted-F1 | F1(yes) | F1(no) | F1(maybe) |")
print("|---|---|---|---|---|---|---|---|")
for r in rows:
    p = r["per_class_f1"]
    print(f"| {r['tag']} | {r['params']} | {r['accuracy']:.4f} | {r['macro_f1']:.4f} | "
          f"{r['weighted_f1']:.4f} | {p[0]:.3f} | {p[1]:.3f} | {p[2]:.3f} |")

os.makedirs("results", exist_ok=True)
with open(f"results/results_qwen_{RUN_MODE}.json", "w", encoding="utf-8") as f:
    json.dump({"model": MODEL_NAME, "run_mode": RUN_MODE, "seed": RANDOM_SEED,
               "epochs": EPOCHS, "lr": LR, "select_by": SELECT_BY,
               "selected_epoch": best_epoch,
               "split": {"train": len(train_idx), "val": len(val_idx), "test": len(test_idx)},
               "val_history": history, "comparison": rows}, f, indent=2, ensure_ascii=False)
print(f"\n已保存 results/results_qwen_{RUN_MODE}.json")


| 模型 | 参数量 | accuracy | macro-F1 | weighted-F1 | F1(yes) | F1(no) | F1(maybe) |
|---|---|---|---|---|---|---|---|
| PubMedBERT baseline (110M, 判别式) | 110M | 0.6533 | 0.4439 | 0.6119 | 0.746 | 0.586 | 0.000 |
| PubMedBERT improved (110M, 判别式) | 110M | 0.6533 | 0.4856 | 0.6252 | 0.730 | 0.615 | 0.111 |
| Qwen2.5-0.5B-Instruct 未微调 (生成式) | 0.5B | 0.1067 | 0.0643 | 0.0206 | 0.000 | 0.000 | 0.193 |
| Qwen2.5-0.5B-Instruct + LoRA-SFT [improved] (生成式) | 0.5B | 0.6133 | 0.4513 | 0.6125 | 0.713 | 0.641 | 0.000 |

已保存 results/results_qwen_improved.json


In [16]:
model.save_pretrained(f"qwen-pubmedqa-lora-{RUN_MODE}")
tok.save_pretrained(f"qwen-pubmedqa-lora-{RUN_MODE}")
!du -sh qwen-pubmedqa-lora-*

45M	qwen-pubmedqa-lora-improved
